# Lesson 8: Evaluating LLM Applications

Welcome to Lesson 8! We have built chains, RAG pipelines, and agents. But how do you know if they actually work **well**? In traditional software, you write unit tests with exact expected outputs. With LLMs, outputs are non-deterministic -- the same input can produce different (but equally valid) outputs.

### The Session Goal
Today, we will learn systematic approaches to **evaluate** LLM applications. We will build automated scoring pipelines that measure quality, detect regressions, and give you confidence before deploying to production.

### The Core Concepts
1. **The Evaluation Problem**: Why `assert output == expected` does not work for LLMs.
2. **LLM-as-Judge**: Using a model to grade another model's output.
3. **Criteria-Based Scoring**: Measuring correctness, relevance, faithfulness, and toxicity.
4. **RAG Evaluation**: Testing retrieval quality separately from generation quality.
5. **Regression Testing**: Building evaluation datasets to catch quality drops over time.

### Why This Matters
- A RAG system that retrieves the wrong chunks will hallucinate confidently
- A prompt change that improves one case might break ten others
- Without evaluation, you are deploying blind

In [ ]:
!pip install -q langchain-core langchain-openai langchain-community langchain-text-splitters chromadb

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Step 1: The Evaluation Problem

In traditional software testing, you compare output to a known correct answer:
```python
assert add(2, 3) == 5  # Deterministic, exact match
```

With LLMs, the same question can produce many valid answers:
- "The capital of France is Paris."
- "Paris is the capital of France."
- "It's Paris."

All three are correct, but none are string-equal. We need **semantic evaluation** -- judging meaning, not characters.

Let's build a simple chain and see why naive testing fails.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Build a simple Q&A chain
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
prompt = ChatPromptTemplate.from_template("Answer this question concisely: {question}")
chain = prompt | model | StrOutputParser()

# Run the same question multiple times
question = "What is the capital of France?"
expected = "The capital of France is Paris."

print("--- Why Exact Match Fails ---\n")
for i in range(3):
    answer = chain.invoke({"question": question})
    exact_match = answer == expected
    print(f"  Run {i+1}: '{answer}'")
    print(f"         Exact match: {exact_match}")
    print()

print("All answers are correct, but exact string matching gives inconsistent results.")
print("We need a smarter evaluation approach.")

---

## Step 2: LLM-as-Judge -- Using a Model to Grade Another Model

The modern solution is to use a **stronger or equal LLM as a judge**. We give it the question, the expected answer, and the actual output, then ask it to score the response.

This is the same pattern used by evaluation frameworks like LangSmith, RAGAS, and DeepEval.

### The Evaluation Chain:
```text
{question, expected_answer, actual_output}
   |---> Evaluation Prompt (asks: "Is this correct?")
         |---> Judge LLM (returns structured score + reasoning)
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Build the judge evaluation chain
judge_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

eval_prompt = ChatPromptTemplate.from_template("""You are an evaluation judge. Grade the following AI response.

Question: {question}
Expected Answer: {expected}
Actual Response: {actual}

Score the response on a scale of 1-5:
  5 = Perfect, semantically identical to expected
  4 = Correct with minor phrasing differences
  3 = Partially correct, missing some information
  2 = Mostly incorrect but shows some understanding
  1 = Completely wrong or irrelevant

Respond in this exact format:
Score: [1-5]
Reasoning: [one sentence explanation]""")

judge_chain = eval_prompt | judge_model | StrOutputParser()

# 2. Test cases
test_cases = [
    {"question": "What is the capital of France?", "expected": "Paris", "actual": "The capital of France is Paris."},
    {"question": "What is the capital of France?", "expected": "Paris", "actual": "Paris is a beautiful city in Europe."},
    {"question": "What is 2+2?", "expected": "4", "actual": "The answer is 5."},
]

print("--- LLM-as-Judge Evaluation ---\n")
for tc in test_cases:
    result = judge_chain.invoke(tc)
    print(f"  Q: {tc['question']}")
    print(f"  Expected: {tc['expected']}")
    print(f"  Actual:   {tc['actual']}")
    print(f"  {result}")
    print()

---

## Step 3: Multi-Criteria Evaluation

A single "correctness" score is not enough for production. You need to evaluate multiple dimensions independently:

| Criterion | What It Measures | Example Failure |
|-----------|-----------------|-----------------|
| **Correctness** | Is the answer factually right? | "Paris is in Germany" |
| **Relevance** | Does it answer the actual question? | Answering a different question |
| **Conciseness** | Is it appropriately brief? | A 500-word answer to "What is 2+2?" |
| **Harmfulness** | Is the output safe and appropriate? | Toxic or biased language |

Let's build a multi-criteria evaluator that scores each dimension separately.

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)

multi_criteria_prompt = ChatPromptTemplate.from_template("""Evaluate the following AI response on multiple criteria.

Question: {question}
AI Response: {response}

Score each criterion from 1 (worst) to 5 (best). Respond ONLY with valid JSON:
{{
  "correctness": {{"score": <1-5>, "reason": "<brief explanation>"}},
  "relevance": {{"score": <1-5>, "reason": "<brief explanation>"}},
  "conciseness": {{"score": <1-5>, "reason": "<brief explanation>"}},
  "harmfulness": {{"score": <1-5>, "reason": "<5 means safe, 1 means harmful>"}}
}}""")

multi_eval_chain = multi_criteria_prompt | judge | StrOutputParser()

# Test with different quality responses
test_responses = [
    {
        "question": "What causes rain?",
        "response": "Rain is caused by water vapor in the atmosphere condensing into droplets that become heavy enough to fall."
    },
    {
        "question": "What causes rain?",
        "response": "Rain happens because of complex meteorological phenomena involving the hydrological cycle, evapotranspiration rates, adiabatic cooling processes, cloud condensation nuclei, and various atmospheric dynamics including but not limited to orographic lift, convective activity, and frontal boundaries between air masses of differing temperatures and humidity levels."
    },
    {
        "question": "What causes rain?",
        "response": "The sun causes rain because it is hot."
    },
]

print("--- Multi-Criteria Evaluation ---\n")
for tc in test_responses:
    result = multi_eval_chain.invoke(tc)
    print(f"Q: {tc['question']}")
    print(f"A: {tc['response'][:80]}...")
    scores = json.loads(result)
    for criterion, data in scores.items():
        print(f"  {criterion:12s}: {data['score']}/5 -- {data['reason']}")
    print()

---

## Step 4: RAG-Specific Evaluation (Retrieval + Generation)

RAG systems have a unique challenge: failures can come from **two independent stages**:
1. **Retrieval failure**: The wrong chunks were fetched (the answer was never given to the LLM)
2. **Generation failure**: The right chunks were fetched but the LLM misinterpreted them

We must evaluate these separately:

| Metric | What It Tests | Caught Issue |
|--------|--------------|--------------|
| **Context Relevance** | Are the retrieved chunks related to the question? | Bad embeddings or chunking |
| **Faithfulness** | Is the answer supported by the retrieved context? | Hallucination despite good retrieval |
| **Answer Correctness** | Is the final answer right? | End-to-end quality check |

Let's build a RAG pipeline and evaluate it at each stage.

In [ ]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# 1. Build a small RAG system (same as Lesson 6)
documents = [
    Document(page_content="Nexora Technologies offers 25 days of paid vacation per year. Unused days can be carried over up to 10 days.", metadata={"source": "vacation_policy"}),
    Document(page_content="Remote work is allowed up to 3 days per week. Core hours are 10am-4pm. Equipment allowance is $1500 annually.", metadata={"source": "remote_policy"}),
    Document(page_content="Performance reviews happen in June and December. Promotion requires two consecutive 'Exceeds Expectations' ratings.", metadata={"source": "performance_policy"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="eval_test")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_template("""Answer based ONLY on this context:
{context}

Question: {question}
Answer:""")

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print("RAG pipeline built. Ready for evaluation.")

In [ ]:
import json

# 2. RAG Evaluation Chain -- evaluates retrieval AND generation quality
rag_eval_prompt = ChatPromptTemplate.from_template("""You are evaluating a RAG (Retrieval-Augmented Generation) system.

Question: {question}
Retrieved Context: {context}
Generated Answer: {answer}
Ground Truth: {ground_truth}

Evaluate and return ONLY valid JSON:
{{
  "context_relevance": {{"score": <1-5>, "reason": "<Are the retrieved chunks relevant to the question?>"}},
  "faithfulness": {{"score": <1-5>, "reason": "<Is the answer supported by the context? No hallucination?>"}},
  "answer_correctness": {{"score": <1-5>, "reason": "<Does the answer match the ground truth?>"}}
}}""")

rag_eval_chain = rag_eval_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

# 3. Define evaluation dataset with ground truths
eval_dataset = [
    {"question": "How many vacation days do employees get?", "ground_truth": "25 days per year"},
    {"question": "How many days can I work remotely?", "ground_truth": "Up to 3 days per week"},
    {"question": "When are performance reviews?", "ground_truth": "June and December"},
    {"question": "What is the CEO's salary?", "ground_truth": "Not available in the knowledge base"},
]

# 4. Run evaluation
print("--- RAG Pipeline Evaluation ---\n")
for item in eval_dataset:
    # Get retrieval results and answer
    retrieved_docs = retriever.invoke(item["question"])
    context = format_docs(retrieved_docs)
    answer = rag_chain.invoke(item["question"])

    # Evaluate
    eval_result = rag_eval_chain.invoke({
        "question": item["question"],
        "context": context,
        "answer": answer,
        "ground_truth": item["ground_truth"]
    })

    scores = json.loads(eval_result)
    print(f"Q: {item['question']}")
    print(f"A: {answer}")
    print(f"  Context Relevance:  {scores['context_relevance']['score']}/5")
    print(f"  Faithfulness:       {scores['faithfulness']['score']}/5")
    print(f"  Answer Correctness: {scores['answer_correctness']['score']}/5")
    print()

---

## Step 5: Regression Testing -- Catching Quality Drops

In production, you will frequently change prompts, swap models, or update your document corpus. Each change risks breaking something that used to work.

The solution is a **golden dataset** -- a fixed set of question/answer pairs that you run after every change. If scores drop below a threshold, the change is rejected.

This is the LLM equivalent of a CI/CD test suite.

In [ ]:
import json

# 1. Define a golden evaluation dataset
golden_dataset = [
    {"question": "How many vacation days per year?", "expected": "25 days"},
    {"question": "Can I carry over unused vacation?", "expected": "Yes, up to 10 days"},
    {"question": "What are core working hours for remote?", "expected": "10am to 4pm"},
    {"question": "How often are reviews conducted?", "expected": "Twice a year, June and December"},
    {"question": "What rating is needed for promotion?", "expected": "Two consecutive Exceeds Expectations"},
]

# 2. Simple correctness evaluator
correctness_prompt = ChatPromptTemplate.from_template("""Grade whether the actual answer is semantically equivalent to the expected answer.

Question: {question}
Expected: {expected}
Actual: {actual}

Return ONLY a JSON object: {{"pass": true/false, "score": <1-5>, "reason": "<brief>"}}""")

correctness_chain = correctness_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

# 3. Run the regression suite
print("--- Regression Test Suite ---\n")
results = []
for item in golden_dataset:
    answer = rag_chain.invoke(item["question"])
    eval_result = correctness_chain.invoke({
        "question": item["question"],
        "expected": item["expected"],
        "actual": answer
    })
    score_data = json.loads(eval_result)
    results.append(score_data)
    status = "PASS" if score_data["pass"] else "FAIL"
    print(f"  [{status}] {item['question']}")
    print(f"         Expected: {item['expected']}")
    print(f"         Got:      {answer}")
    print(f"         Score: {score_data['score']}/5\n")

# 4. Summary metrics
total = len(results)
passed = sum(1 for r in results if r["pass"])
avg_score = sum(r["score"] for r in results) / total

print(f"{'='*50}")
print(f"Results: {passed}/{total} passed ({passed/total*100:.0f}%)")
print(f"Average Score: {avg_score:.1f}/5")
print(f"Threshold: 80% pass rate required for deployment")
print(f"Status: {'DEPLOY' if passed/total >= 0.8 else 'BLOCKED'}")

---

## Step 6: Comparing Model Versions Side-by-Side

When upgrading models (e.g., gpt-4o-mini to gpt-4o), you need to verify the new model performs at least as well. Run both through the same evaluation dataset and compare scores directly.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import json

# 1. Build two chains with different model configurations (simulating a version upgrade)
prompt = ChatPromptTemplate.from_template("Answer concisely: {question}")
parser = StrOutputParser()

chain_a = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | parser      # Current
chain_b = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0.9) | parser     # Candidate (higher temp)

# 2. Evaluation judge
compare_prompt = ChatPromptTemplate.from_template("""Compare two AI responses to the same question.

Question: {question}
Expected: {expected}
Response A: {response_a}
Response B: {response_b}

Return ONLY valid JSON:
{{"score_a": <1-5>, "score_b": <1-5>, "winner": "A" or "B" or "tie", "reason": "<brief>"}}""")

compare_chain = compare_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

# 3. Run comparison
comparison_questions = [
    {"question": "What is photosynthesis?", "expected": "Process where plants convert sunlight into energy"},
    {"question": "Name three programming languages", "expected": "Python, JavaScript, Java (or similar)"},
    {"question": "What is the speed of light?", "expected": "Approximately 300,000 km/s"},
]

print("--- Model Comparison: A (temp=0) vs B (temp=0.9) ---\n")
a_wins, b_wins, ties = 0, 0, 0

for item in comparison_questions:
    resp_a = chain_a.invoke(item)
    resp_b = chain_b.invoke(item)
    result = compare_chain.invoke({
        "question": item["question"],
        "expected": item["expected"],
        "response_a": resp_a,
        "response_b": resp_b
    })
    data = json.loads(result)
    print(f"  Q: {item['question']}")
    print(f"  A (score {data['score_a']}): {resp_a[:60]}...")
    print(f"  B (score {data['score_b']}): {resp_b[:60]}...")
    print(f"  Winner: {data['winner']} -- {data['reason']}\n")

    if data["winner"] == "A": a_wins += 1
    elif data["winner"] == "B": b_wins += 1
    else: ties += 1

print(f"Final: Model A wins {a_wins}, Model B wins {b_wins}, Ties {ties}")
print(f"Recommendation: {'Keep A' if a_wins >= b_wins else 'Switch to B'}")

---

## Summary and Key Takeaways

Today we built a complete evaluation framework for LLM applications:

| Technique | Use Case | Key Insight |
|-----------|----------|-------------|
| **LLM-as-Judge** | Score any output semantically | Replaces brittle exact-match testing |
| **Multi-Criteria** | Evaluate correctness, relevance, conciseness independently | A correct answer can still fail on other dimensions |
| **RAG Evaluation** | Test retrieval and generation separately | Diagnoses WHERE failures occur |
| **Regression Suite** | Catch quality drops before deployment | The LLM equivalent of CI/CD tests |
| **Model Comparison** | Validate upgrades and config changes | Data-driven model selection |

### The Evaluation Workflow in Production
```text
1. Build golden dataset (50-200 representative Q&A pairs)
2. Run evaluation after every prompt/model/data change
3. Set pass thresholds (e.g., 80% pass rate, avg score > 4.0)
4. Block deployment if thresholds are not met
5. Continuously expand the dataset with real user failures
```

### Tools in the Ecosystem
- **LangSmith**: LangChain's hosted evaluation and tracing platform
- **RAGAS**: Open-source RAG evaluation framework
- **DeepEval**: Unit testing framework for LLMs
- **Phoenix (Arize)**: Observability and evaluation for ML/LLM systems

### Next Lesson Preview
We have evaluation to measure quality. But what about **preventing harm**? In Lesson 9, we will cover security and guardrails -- protecting your LLM application from prompt injection, jailbreaks, and data leakage.